# 5b. EDA with SQL - Additional Queries
This notebook extends Notebook 5. It runs SQL on the dashboard dataset (56 launches) to cover launch sites, payloads, success rates, rankings and a time analysis. Run all cells in Google Colab.

In [1]:
import pandas as pd
import sqlite3
from IPython.display import display

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
raw = pd.read_csv(url)
launches = raw.rename(columns={'Launch Site': 'LaunchSite', 'Payload Mass (kg)': 'PayloadMass', 'class': 'Class'})[['LaunchSite', 'PayloadMass', 'Class']]
conn = sqlite3.connect(':memory:')
launches.to_sql('LAUNCHES', conn, index=False, if_exists='replace')
print(launches.shape)

(56, 3)


## Query 1 - Launches per launch site

In [2]:
query = """SELECT LaunchSite, COUNT(*) AS Launches FROM LAUNCHES GROUP BY LaunchSite ORDER BY Launches DESC"""
display(pd.read_sql_query(query, conn))

,LaunchSite,Launches
0,CCAFS LC-40,26
1,KSC LC-39A,13
2,VAFB SLC-4E,10
3,CCAFS SLC-40,7


## Query 2 - Successful landings and success rate per site

In [3]:
query = """SELECT LaunchSite, COUNT(*) AS Launches, SUM(Class) AS Successes, ROUND(100.0*SUM(Class)/COUNT(*),1) AS SuccessRatePct FROM LAUNCHES GROUP BY LaunchSite ORDER BY SuccessRatePct DESC"""
display(pd.read_sql_query(query, conn))

,LaunchSite,Launches,Successes,SuccessRatePct
0,KSC LC-39A,13,10,76.9
1,CCAFS SLC-40,7,3,42.9
2,VAFB SLC-4E,10,4,40.0
3,CCAFS LC-40,26,7,26.9


## Query 3 - Rank launch sites by success rate

In [4]:
query = """SELECT RANK() OVER (ORDER BY 1.0*SUM(Class)/COUNT(*) DESC) AS Rank, LaunchSite, ROUND(100.0*SUM(Class)/COUNT(*),1) AS SuccessRatePct FROM LAUNCHES GROUP BY LaunchSite"""
display(pd.read_sql_query(query, conn))

,Rank,LaunchSite,SuccessRatePct
0,1,KSC LC-39A,76.9
1,2,CCAFS SLC-40,42.9
2,3,VAFB SLC-4E,40.0
3,4,CCAFS LC-40,26.9


## Query 4 - Payload mass statistics per site

In [5]:
query = """SELECT LaunchSite, MIN(PayloadMass) AS MinKg, ROUND(AVG(PayloadMass),0) AS AvgKg, MAX(PayloadMass) AS MaxKg FROM LAUNCHES GROUP BY LaunchSite ORDER BY AvgKg DESC"""
display(pd.read_sql_query(query, conn))

,LaunchSite,MinKg,AvgKg,MaxKg
0,VAFB SLC-4E,475.0,5814.0,9600.0
1,KSC LC-39A,2490.0,4377.0,6761.0
2,CCAFS SLC-40,362.0,3517.0,6092.0
3,CCAFS LC-40,0.0,2591.0,5271.0


## Query 5 - Average payload for successful and failed landings

In [6]:
query = """SELECT CASE WHEN Class=1 THEN 'Success' ELSE 'Failure' END AS Outcome, COUNT(*) AS Launches, ROUND(AVG(PayloadMass),0) AS AvgPayloadKg FROM LAUNCHES GROUP BY Class"""
display(pd.read_sql_query(query, conn))

,Outcome,Launches,AvgPayloadKg
0,Failure,32,3476.0
1,Success,24,3991.0


## Query 6 - Success rate by payload band

In [7]:
query = """SELECT CASE WHEN PayloadMass < 2000 THEN '0-1999 kg' WHEN PayloadMass < 4000 THEN '2000-3999 kg' WHEN PayloadMass < 6000 THEN '4000-5999 kg' ELSE '6000 kg and above' END AS PayloadBand,
 COUNT(*) AS Launches, SUM(Class) AS Successes, ROUND(100.0*SUM(Class)/COUNT(*),1) AS SuccessRatePct FROM LAUNCHES GROUP BY PayloadBand ORDER BY MIN(PayloadMass)"""
display(pd.read_sql_query(query, conn))

,PayloadBand,Launches,Successes,SuccessRatePct
0,0-1999 kg,13,3,23.1
1,2000-3999 kg,21,13,61.9
2,4000-5999 kg,13,5,38.5
3,6000 kg and above,9,3,33.3


## Query 7 - Overall totals

In [8]:
query = """SELECT COUNT(*) AS Launches, SUM(Class) AS Successes, COUNT(*)-SUM(Class) AS Failures, ROUND(100.0*SUM(Class)/COUNT(*),1) AS SuccessRatePct, ROUND(AVG(PayloadMass),0) AS AvgPayloadKg, MAX(PayloadMass) AS MaxPayloadKg FROM LAUNCHES"""
display(pd.read_sql_query(query, conn))

,Launches,Successes,Failures,SuccessRatePct,AvgPayloadKg,MaxPayloadKg
0,56,24,32,42.9,3697.0,9600.0


## Query 8 - Time analysis: landing success rate by year
Uses the 90-launch prepared dataset from Notebook 4.

In [9]:
df90 = pd.read_csv('https://raw.githubusercontent.com/chuksoo/IBM-Data-Science-Capstone-SpaceX/main/dataset_part_2.csv')
df90[['Date', 'Class']].to_sql('LAUNCHES90', conn, index=False, if_exists='replace')
query = """SELECT strftime('%Y', Date) AS Year, COUNT(*) AS Launches, ROUND(AVG(Class), 3) AS SuccessRate
FROM LAUNCHES90 GROUP BY Year ORDER BY Year"""
display(pd.read_sql_query(query, conn))

,Year,Launches,SuccessRate
0,2010,1,0.000
1,2012,1,0.000
2,2013,3,0.000
3,2014,6,0.333
4,2015,6,0.333
5,2016,8,0.625
6,2017,18,0.833
7,2018,18,0.611
8,2019,10,0.900
9,2020,19,0.842


## Interpretation
- KSC LC-39A ranks first by landing success rate; CCAFS LC-40 ranks last.
- The 2000-3999 kg payload band has the highest success rate in this dataset.
- Success rate rises strongly across the years (time analysis).